# 02 Recompute the 55 address features

1. Builds the per-transaction table, one row per labelled address and Elliptic transaction it takes part in.
2. Aggregates it to the 55 features, once over all transactions of each address (lifetime set) and once with the
   transactions of each training address restricted to blocks up to the cutoff (leakage-resistant set).
3. Compares the two sets.

The definitions are given in the appendix of the paper. Expected: 367,517 rows in the per-transaction table,
265,352 addresses in each set, and 2,501 training addresses whose features differ between the sets.

In [1]:
from pathlib import Path
import duckdb

ROOT      = Path.cwd().parent
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS   = ROOT / 'outputs'

THREADS      = 192
MEMORY_LIMIT = '320GB'
TEMP_DIR     = PROCESSED / 'duckdb_tmp'

con = duckdb.connect()
con.execute(f'PRAGMA threads={THREADS}')
con.execute(f"PRAGMA memory_limit='{MEMORY_LIMIT}'")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")

import json
import pandas as pd

K_TXIN  = PROCESSED / 'kondor_txin.parquet'
K_TXOUT = PROCESSED / 'kondor_txout.parquet'
K_TX    = PROCESSED / 'kondor_tx.parquet'
ADDRESS_MAP    = PROCESSED / 'address_map.parquet'
ELLIPTIC_TX    = PROCESSED / 'elliptic_tx.parquet'
TIMESTEP_BLOCK = PROCESSED / 'timestep_blocks.parquet'
CUTOFF = json.loads((PROCESSED / 'cutoff.json').read_text())['cutoff_block']

FEATURES_LIFETIME  = PROCESSED / 'features_lifetime.parquet'
FEATURES_RESISTANT = PROCESSED / 'features_leakage_resistant.parquet'

STATS = ['total', 'min', 'max', 'mean', 'median']
FEATURES = (
    [f'btc_{q}_{s}' for q in ['transacted', 'sent', 'received'] for s in STATS]
    + [f'fees_{s}' for s in STATS] + [f'fees_as_share_{s}' for s in STATS]
    + [f'blocks_btwn_{q}_{s}' for q in ['txs', 'input_txs', 'output_txs'] for s in STATS]
    + [f'transacted_w_address_{s}' for s in STATS] + ['num_addr_transacted_multiple']
    + ['total_txs', 'num_txs_as_sender', 'num_txs_as receiver']
    + ['first_block_appeared_in', 'last_block_appeared_in', 'lifetime_in_blocks',
       'first_sent_block', 'first_received_block', 'num_timesteps_appeared_in'])
assert len(FEATURES) == 55
print(f'cutoff block {CUTOFF:,}')

cutoff block 457,734


## Inputs and outputs of the Elliptic transactions

All later queries are restricted to the Elliptic transactions found in the ledger. Amounts are converted from satoshis to bitcoin.

In [2]:
con.execute(f"""
    CREATE OR REPLACE TABLE etx AS SELECT DISTINCT kondor_txID AS txID FROM '{ELLIPTIC_TX}';
    CREATE OR REPLACE TABLE ti AS
        SELECT txID, addrID, sum AS sats FROM '{K_TXIN}' WHERE txID IN (SELECT txID FROM etx);
    CREATE OR REPLACE TABLE tout AS
        SELECT txID, addrID, sum AS sats FROM '{K_TXOUT}' WHERE txID IN (SELECT txID FROM etx);
    CREATE OR REPLACE TABLE egos AS
        SELECT addrID, address, label, split FROM '{ADDRESS_MAP}';
    CREATE OR REPLACE TABLE tx_totals AS
        SELECT t.txID, t.blockID,
               COALESCE(i.s, 0) / 1e8 AS tx_in,
               COALESCE(o.s, 0) / 1e8 AS tx_out,
               GREATEST(COALESCE(i.s, 0) - COALESCE(o.s, 0), 0) / 1e8 AS tx_fee
        FROM (SELECT txID, blockID FROM '{K_TX}' WHERE txID IN (SELECT txID FROM etx)) t
        LEFT JOIN (SELECT txID, SUM(sats) AS s FROM ti   GROUP BY txID) i USING (txID)
        LEFT JOIN (SELECT txID, SUM(sats) AS s FROM tout GROUP BY txID) o USING (txID);
""")
print(con.execute('SELECT COUNT(*) FROM etx').fetchone()[0], 'Elliptic transactions')

202804 Elliptic transactions


## Per-transaction table

For each labelled address and transaction, `ego_in` and `ego_out` are the amounts the address contributes as inputs and
collects as outputs. The net amount sent is `max(ego_in - ego_out, 0)`, the net amount received is `max(ego_out - ego_in, 0)`,
and the amount transacted is `|ego_in - ego_out|`. The fee share attributed to the address is the transaction fee times
`max(ego_in, ego_out)` divided by the total input of the transaction.

In [3]:
con.execute("""
    CREATE OR REPLACE TABLE per_tx AS
    WITH ein AS (
        SELECT addrID, txID, SUM(sats) / 1e8 AS ego_in FROM ti
        WHERE addrID IN (SELECT addrID FROM egos) GROUP BY addrID, txID),
    eout AS (
        SELECT addrID, txID, SUM(sats) / 1e8 AS ego_out FROM tout
        WHERE addrID IN (SELECT addrID FROM egos) GROUP BY addrID, txID),
    stake AS (
        SELECT COALESCE(ein.addrID, eout.addrID) AS addrID,
               COALESCE(ein.txID, eout.txID) AS txID,
               COALESCE(ein.ego_in, 0) AS ego_in,
               COALESCE(eout.ego_out, 0) AS ego_out
        FROM ein FULL OUTER JOIN eout ON ein.addrID = eout.addrID AND ein.txID = eout.txID)
    SELECT s.addrID, s.txID, t.blockID,
           GREATEST(s.ego_in - s.ego_out, 0) AS net_sent,
           GREATEST(s.ego_out - s.ego_in, 0) AS net_received,
           ABS(s.ego_in - s.ego_out) AS transacted,
           t.tx_fee,
           CASE WHEN t.tx_in > 0 THEN t.tx_fee * GREATEST(s.ego_in, s.ego_out) / t.tx_in ELSE 0 END AS fee_share
    FROM stake s JOIN tx_totals t USING (txID)""")
print(con.execute('SELECT COUNT(*) FROM per_tx').fetchone()[0], 'rows in the per-transaction table')

367517 rows in the per-transaction table


## Aggregation

`aggregate(rows)` computes the 55 features from a view of the per-transaction table. Block gaps are the differences
between the blocks of consecutive transactions, over all, sending, and receiving transactions, and are zero for an
address with fewer than two such transactions. The counterparties of an address in a transaction are the output addresses
when it sends and the input addresses when it receives, excluding the address itself.

In [4]:
def aggregate(rows):
    def stat_block(col, prefix):
        return ', '.join(f'{fn}({col}) AS {prefix}_{s}'
                         for fn, s in zip(['SUM', 'MIN', 'MAX', 'AVG', 'MEDIAN'], STATS))

    gaps = {}
    for name, cond in [('txs', 'TRUE'), ('input_txs', 'net_sent > 0'), ('output_txs', 'net_received > 0')]:
        gaps[name] = f"""
            SELECT addrID, {stat_block('g', f'blocks_btwn_{name}')}
            FROM (SELECT addrID, blockID - LAG(blockID) OVER (PARTITION BY addrID ORDER BY blockID) AS g
                  FROM {rows} WHERE {cond})
            WHERE g IS NOT NULL GROUP BY addrID"""

    query = f"""
    WITH base AS (
        SELECT addrID,
               {stat_block('transacted', 'btc_transacted')},
               {stat_block('net_sent', 'btc_sent')},
               {stat_block('net_received', 'btc_received')},
               {stat_block('tx_fee', 'fees')},
               {stat_block('fee_share', 'fees_as_share')},
               COUNT(*) AS total_txs,
               SUM(CASE WHEN net_sent > 0 THEN 1 ELSE 0 END) AS num_txs_as_sender,
               SUM(CASE WHEN net_received > 0 THEN 1 ELSE 0 END) AS "num_txs_as receiver",
               MIN(blockID) AS first_block_appeared_in,
               MAX(blockID) AS last_block_appeared_in,
               MAX(blockID) - MIN(blockID) AS lifetime_in_blocks,
               COALESCE(MIN(CASE WHEN net_sent > 0 THEN blockID END), 0) AS first_sent_block,
               COALESCE(MIN(CASE WHEN net_received > 0 THEN blockID END), 0) AS first_received_block
        FROM {rows} GROUP BY addrID),
    g_all AS ({gaps['txs']}), g_in AS ({gaps['input_txs']}), g_out AS ({gaps['output_txs']}),
    steps AS (
        SELECT d.addrID, COUNT(DISTINCT s.time_step) AS num_timesteps_appeared_in
        FROM (SELECT DISTINCT addrID, blockID FROM {rows}) d
        JOIN '{TIMESTEP_BLOCK}' s ON d.blockID BETWEEN s.first_block AND s.last_block
        GROUP BY d.addrID),
    sides AS (
        SELECT addrID AS ego, txID,
               MAX(CASE WHEN net_sent > 0 THEN 1 ELSE 0 END) AS sends,
               MAX(CASE WHEN net_received > 0 THEN 1 ELSE 0 END) AS receives
        FROM {rows} GROUP BY addrID, txID),
    pairs AS (
        SELECT s.ego, s.txID, o.addrID AS cp
        FROM sides s JOIN (SELECT DISTINCT txID, addrID FROM tout) o USING (txID)
        WHERE s.sends = 1 AND o.addrID <> s.ego
        UNION
        SELECT s.ego, s.txID, i.addrID AS cp
        FROM sides s JOIN (SELECT DISTINCT txID, addrID FROM ti) i USING (txID)
        WHERE s.receives = 1 AND i.addrID <> s.ego),
    cp AS (
        SELECT ego AS addrID, {stat_block('n', 'transacted_w_address')},
               SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS num_addr_transacted_multiple
        FROM (SELECT ego, cp, COUNT(DISTINCT txID) AS n FROM pairs GROUP BY ego, cp) GROUP BY ego)
    SELECT e.address, e.addrID, e.label, e.split, base.* EXCLUDE (addrID),
           g_all.* EXCLUDE (addrID), g_in.* EXCLUDE (addrID), g_out.* EXCLUDE (addrID),
           steps.num_timesteps_appeared_in, cp.* EXCLUDE (addrID)
    FROM egos e
    JOIN base USING (addrID)
    LEFT JOIN g_all USING (addrID) LEFT JOIN g_in USING (addrID) LEFT JOIN g_out USING (addrID)
    LEFT JOIN steps USING (addrID) LEFT JOIN cp USING (addrID)
    ORDER BY e.address"""
    df = con.execute(query).fetchdf()
    df[FEATURES] = df[FEATURES].fillna(0)
    return df[['address', 'addrID', 'label', 'split'] + FEATURES]

## Lifetime and leakage-resistant sets

The lifetime set uses every Elliptic transaction of each address and reproduces the construction of Elliptic++.
The leakage-resistant set keeps, for each training address, only its transactions in blocks up to the cutoff.
Test addresses first appear after the cutoff, so their rows are the same in both sets.

In [5]:
con.execute("CREATE OR REPLACE VIEW rows_lifetime AS SELECT * FROM per_tx")
con.execute(f"""
    CREATE OR REPLACE VIEW rows_resistant AS
    SELECT p.* FROM per_tx p JOIN egos e USING (addrID)
    WHERE e.split = 'test' OR p.blockID <= {CUTOFF}""")

lifetime  = aggregate('rows_lifetime')
resistant = aggregate('rows_resistant')
lifetime.to_parquet(FEATURES_LIFETIME, index=False)
resistant.to_parquet(FEATURES_RESISTANT, index=False)
print(f'lifetime {lifetime.shape}  leakage-resistant {resistant.shape}')

lifetime (265352, 59)  leakage-resistant (265352, 59)


## Effect of the cutoff

In [6]:
assert (lifetime.address == resistant.address).all()
changed = (lifetime[FEATURES].to_numpy() != resistant[FEATURES].to_numpy()).any(axis=1)
summary = (pd.DataFrame({'split': lifetime.split, 'label': lifetime.label, 'changed': changed})
           .groupby(['split', 'label']).changed.agg(['sum', 'count', 'mean']))
print(summary.to_string())
train = lifetime.split == 'train'
print(f'training addresses changed {changed[train].sum():,} of {train.sum():,} ({changed[train].mean():.2%})')
print(f'test addresses changed {changed[~train].sum():,}')

              sum   count      mean
split label                        
test  1         5    4889  0.001023
      2       173   87560  0.001976
train 1        66    9377  0.007038
      2      3021  163526  0.018474
training addresses changed 3,087 of 172,903 (1.79%)
test addresses changed 178
